In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
from langchain.chat_models import init_chat_model

model = init_chat_model(model="groq:qwen/qwen3-32b")
model.invoke("hi")

c:\Users\mendh\Langchain-Langgraph\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


AIMessage(content='<think>\nOkay, the user said "hi". That\'s a simple greeting. I need to respond in a friendly and welcoming way. Maybe start with a greeting of my own and offer assistance. Let me check if there are any specific guidelines for how to handle such interactions. The user might be looking for help with something, but since they just said hi, I should keep it open-ended. Let me make sure the response is positive and encourages them to ask for help if needed. Also, keep it concise but warm.\n</think>\n\nHello! How can I assist you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 119, 'prompt_tokens': 9, 'total_tokens': 128, 'completion_time': 0.218222291, 'completion_tokens_details': None, 'prompt_time': 0.000294736, 'prompt_tokens_details': None, 'queue_time': 0.046787374, 'total_time': 0.218517027}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs

In [3]:
agent = create_agent(
    model = 'groq:llama-3.1-8b-instant',
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = 'groq:llama-3.1-8b-instant',
            trigger = ("messages",10),
            keep = ("messages",4)
        )
    ]
)

In [4]:
config = {"configurable":{"thread_id":"1"}}

questions = [
    "what is 2+2?",
    "what is 10*5?",
    "what is 1000/4",
    "what is 15-7",
    "what is 3*3+1",
    "what is 4/4*1+1-2"
]

for question in questions:
    print(question)
    response = agent.invoke({"messages":[HumanMessage(content=question)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response["messages"])}")

what is 2+2?
Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='58399050-f85d-446b-8366-cc7a78f5ba9a'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.005814946, 'completion_tokens_details': None, 'prompt_time': 0.005522791, 'prompt_tokens_details': None, 'queue_time': 0.081895249, 'total_time': 0.011337737}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d61bb-6dac-7552-a416-5cfb27d03592-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
Messages: 2
what is 10*5?
Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='58399050-f85d-446b-836

### Human in the loop MiddleWare

In [5]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

def read_email_tool(email_id):
    """Function to read an email by its mail id"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipent,subject,body):
    """Function to send an email"""
    return f"Email sent to {recipent} with subject '{subject} and body {body}'"

In [6]:
agent = create_agent(
    model='groq:llama-3.1-8b-instant',
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email_tool":{
                "allowed_decisions":["approve","edit","reject"]
            },
            "read_email_tool":False,
        }
    )]
)

In [7]:
config = {"configurable":{"thread_id":"test-approve"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="send email to 'john@test.com' with subject 'Hello' and body 'How are you?'")]},
    config=config
)

result

{'messages': [HumanMessage(content="send email to 'john@test.com' with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='2f8d6a9e-4fa6-468c-9430-302e70e0f1c2'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6ff0r5z74', 'function': {'arguments': '{"body":"How are you?","recipent":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 305, 'total_tokens': 339, 'completion_time': 0.037228188, 'completion_tokens_details': None, 'prompt_time': 0.023173475, 'prompt_tokens_details': None, 'queue_time': 0.045795643, 'total_time': 0.060401663}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d61bb-75a4-7883-9d06-9f5150a5e681-0', tool_calls=[{'name': 'send_email_tool', 'args': {'bod

### Approve

In [8]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

print(f"Result: {result["messages"][-1].content}")

paused! Approving...
Result: The response from the function call doesn't contain a message indicating that the email was sent successfully, so we can't be sure that the email was sent. However, based on the function name and parameters, it is likely that the email was sent.


In [9]:
result

{'messages': [HumanMessage(content="send email to 'john@test.com' with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='2f8d6a9e-4fa6-468c-9430-302e70e0f1c2'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6ff0r5z74', 'function': {'arguments': '{"body":"How are you?","recipent":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 305, 'total_tokens': 339, 'completion_time': 0.037228188, 'completion_tokens_details': None, 'prompt_time': 0.023173475, 'prompt_tokens_details': None, 'queue_time': 0.045795643, 'total_time': 0.060401663}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d61bb-75a4-7883-9d06-9f5150a5e681-0', tool_calls=[{'name': 'send_email_tool', 'args': {'bod

### Reject

In [10]:
config = {"configurable":{"thread_id":"test-reject"}}

result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to Vishal@gamil.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)



In [11]:
result

{'messages': [HumanMessage(content="Send email to Vishal@gamil.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='ce764028-525f-4f2e-b3f8-463f79542f50'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'me16m54yb', 'function': {'arguments': '{"body":"How are you?","recipent":"Vishal@gamil.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 305, 'total_tokens': 342, 'completion_time': 0.047673064, 'completion_tokens_details': None, 'prompt_time': 0.479404283, 'prompt_tokens_details': None, 'queue_time': 0.111909947, 'total_time': 0.527077347}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d61bb-78d9-7b90-8e27-7de2620b82ea-0', tool_calls=[{'name': 'send_email_tool', 'args': {

In [12]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )

print(f"Result: {result["messages"][-1].content}")

paused! Approving...
Result: Sorry to hear that the tool call was rejected. Let me try again, this time with more caution.

To send an email, we need to ensure that we have the correct recipient's email address and the email's subject and body. In this case, we have:

- Recipient's email address: Vishal@gamil.com
- Email subject: Hello
- Email body: How are you?

Before sending the email, let's verify the recipient's email address. We can use the `read_email_tool` function to fetch the email's metadata, including the sender's email address.

If the recipient's email address is correct, we can proceed with sending the email using the `send_email_tool` function.

Here's the corrected code:




In [13]:
result

{'messages': [HumanMessage(content="Send email to Vishal@gamil.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='ce764028-525f-4f2e-b3f8-463f79542f50'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'me16m54yb', 'function': {'arguments': '{"body":"How are you?","recipent":"Vishal@gamil.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 305, 'total_tokens': 342, 'completion_time': 0.047673064, 'completion_tokens_details': None, 'prompt_time': 0.479404283, 'prompt_tokens_details': None, 'queue_time': 0.111909947, 'total_time': 0.527077347}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d61bb-78d9-7b90-8e27-7de2620b82ea-0', tool_calls=[{'name': 'send_email_tool', 'args': {